## 学習に用いる骨格のモーションデータを保存する
カメラ2台バージョン

In [1]:
import numpy as np
import datetime
import time
# Poseの各ランドマークを保存
class body1():
    def __init__(self, name_landmarks, dirName="motion"):
        self.dirName = dirName
        self.fileName = "motion"
        self.vec_size = 33
        
        # 以下は触らない
        self.name_landmarks = name_landmarks
        self.isRecord = False
        self.data = np.empty((0,self.vec_size,3), dtype=np.float32)
        self.time_ary = []
        
        
    def record1(self, results):
        if self.isRecord == False:
            return False
        if results.pose_landmarks is None:
            return False
        
        data = np.empty((self.vec_size, 3), dtype=np.float32)
        for i in range(self.vec_size):
            data[i] = LandmarkToArray(results.pose_landmarks.landmark[i])

        self.data = np.vstack([self.data, np.array([data])])
        self.time_ary.append(time.perf_counter())
        return True
        
    
    def record2(self, results_front, results_sagittal):
        if self.isRecord == False:
            return False
        if results_front.pose_landmarks is None:
            return False
        if results_sagittal.pose_landmarks is None:
            return False
        
        data = np.empty((self.vec_size, 3), dtype=np.float32)
        for i in range(self.vec_size):
            front = LandmarkToArray(results_front.pose_landmarks.landmark[i])
            sagittal = LandmarkToArray(results_sagittal.pose_landmarks.landmark[i])
            data[i][0] = front[0]    # x座標は前額面から
            #data[i][1] = (front[1] + sagittal[1]) / 2    # y座標は矢状面と前額面のy座標の平均
            data[i][1] = sagittal[1]    # y座標はとりあえず矢状面から取得
            data[i][2] = sagittal[0]    # z座標は矢状面から

        self.data = np.vstack([self.data, np.array([data])])
        self.time_ary.append(time.perf_counter())
        return True
    
    def save(self, data_label):
        # numpy行列を保存
        data = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        filename_data = self.dirName + "/" + self.fileName + data
        #np.save(filename_data, self.data)
        print(filename_data)
        print(self.data.shape)
        print(self.data)
        # CSVに保存
        print(len(self.time_ary))
        for i in range(len(self.time_ary) - 1):
            self.time_ary[i+1] = self.time_ary[i+1] - self.time_ary[0]
        self.time_ary[0] = 0.0
        frame_ary = list(range(len(self.time_ary)))
        label1 = np.array(data_label * 3)
        label1 = label1.reshape([3, self.vec_size]).reshape(-1, order='F')
        label2 = ['X', 'Y', 'Z'] * self.vec_size
        label = "Frame,Time," + ",".join(label1) + "\n" + ",," + ",".join(label2)
        csv_data = self.data.reshape([self.data.shape[0], 3*self.vec_size])
        csv_data = np.insert(csv_data, 0, self.time_ary, axis=1)
        csv_data = np.insert(csv_data, 0, frame_ary, axis=1)
        np.savetxt(filename_data + ".csv", csv_data, delimiter=',', 
                   header=label, comments='',
                   fmt='%f')
        
    def start_stop(self):
        if self.isRecord == False:
            self.isRecord = True
            clear_output(wait=True)
            print("Start")
            return
        else:
            label = [e.name for e in self.name_landmarks]
            self.save(label)
            print("Recorded")
            self.data = np.empty((0,self.vec_size,3), dtype=np.float32)
            self.time_ary = []
            self.isRecord = False
            return

## """ 学習 """
''' ポーズを取得 '''
# https://google.github.io/mediapipe/solutions/holistic

import cv2
import mediapipe as mp
from IPython.display import clear_output


# NormalizedLandmarkをnp.arrayに変換する
def LandmarkToArray(landmark):
    return np.array([landmark.x, landmark.y, landmark.z])


from pickle import FALSE
import cv2
import mediapipe as mp 

''' 顔とポーズを取得 '''
# https://google.github.io/mediapipe/solutions/holistic
class my_mediapipe():

    def __init__(self, capture=0, window_name="MediaPipe"):
        self.windowName = window_name
        # For webcam input:
        self.cap = cv2.VideoCapture(capture)

        self.mp_drawing = mp.solutions.drawing_utils
        self.mp_drawing_styles = mp.solutions.drawing_styles
        self.mp_holistic = mp.solutions.holistic

        self.mp_drawing = mp.solutions.drawing_utils
        self.mesh_drawing_spec = self.mp_drawing.DrawingSpec(thickness=2,  color=(0,255,0))
        self.mark_drawing_spec = self.mp_drawing.DrawingSpec(thickness=3,  circle_radius=3, color=(0,0,255))

        self.holistic = self.mp_holistic.Holistic(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5)


    # 終了時に必ず呼び出し
    def close(self):
        self.holistic.close()    # withを使わない代わり？
        self.cap.release()
        #cv2.destroyAllWindows()
        
        
    def get_landmarks(self):
        return self.mp_holistic.PoseLandmark


    # with文が無くても動く？
    def loop(self):
        if self.cap.isOpened():
            success, image = self.cap.read()
            if not success:
                print("Ignoring empty camera frame.")
                # If loading a video, use 'break' instead of 'continue'.
                return

            # To improve performance, optionally mark the image as not writeable to
            # pass by reference.
            image.flags.writeable = False
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            results = self.holistic.process(image)

            # Draw landmark annotation on the image.
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            '''
            self.mp_drawing.draw_landmarks(    # 顔の特徴
                image,
                results.face_landmarks,
                self.mp_holistic.FACEMESH_CONTOURS,
                landmark_drawing_spec=None,
                connection_drawing_spec=self.mp_drawing_styles
                .get_default_face_mesh_contours_style())
            '''
            self.mp_drawing.draw_landmarks(    # ポーズの特徴
                image,
                results.pose_landmarks,
                self.mp_holistic.POSE_CONNECTIONS,
                landmark_drawing_spec=self.mp_drawing_styles
                .get_default_pose_landmarks_style())
            '''
            # 以下サンプルプログラムに追加
            self.mp_drawing.draw_landmarks(    # 顔
                image,
                results.face_landmarks,
                self.mp_holistic.FACEMESH_TESSELATION,
                landmark_drawing_spec=None,
                connection_drawing_spec=self.mp_drawing_styles
                .get_default_face_mesh_tesselation_style())

            self.mp_drawing.draw_landmarks(    # 左手
                image,
                results.left_hand_landmarks,
                self.mp_holistic.HAND_CONNECTIONS,
                landmark_drawing_spec = self.mark_drawing_spec,
                connection_drawing_spec = self.mesh_drawing_spec
            )
            self.mp_drawing.draw_landmarks(    # 右手
                image,
                results.right_hand_landmarks,
                self.mp_holistic.HAND_CONNECTIONS,
                landmark_drawing_spec = self.mark_drawing_spec,
                connection_drawing_spec = self.mesh_drawing_spec
            )
            '''
            # Flip the image horizontally for a selfie-view display.
            cv2.imshow(self.windowName, cv2.flip(image, 1))

            return results

        else:
            return None

In [4]:
mediapipe1 = my_mediapipe(0, "Mediapipe (Frontal)")
#mediapipe2 = my_mediapipe("t1.mp4", "Mediapipe (Sagittal)")
learnData = body1(mediapipe1.get_landmarks(), "stop")

while(True):
    result_frontal = mediapipe1.loop()
    #result_sagittal = mediapipe2.loop()
    key = cv2.waitKey(1) & 0xFF
    if key == 27:    # Escで終了
        break;
    if key == ord(' '):    # Spaceキーで録画・終了
        learnData.start_stop()

    learnData.record1(result_frontal)
    #learnData.record2(result_frontal, result_sagittal)

mediapipe1.close()
#mediapipe2.close()
cv2.destroyAllWindows()

Start
stop/motion20230227_005330
(161, 33, 3)
[[[ 0.5198621   0.3126539  -0.8014124 ]
  [ 0.54487747  0.24688803 -0.7542598 ]
  [ 0.5622204   0.24716888 -0.7541028 ]
  ...
  [ 0.4062077   2.6775863   0.9666402 ]
  [ 0.6331922   2.7549942   0.4793318 ]
  [ 0.4557454   2.7673662   0.40457767]]

 [[ 0.519612    0.3132977  -0.8094962 ]
  [ 0.54459685  0.24733628 -0.76251537]
  [ 0.5620604   0.24765915 -0.76234853]
  ...
  [ 0.39949614  2.6764603   0.9794195 ]
  [ 0.62748045  2.7547927   0.50812924]
  [ 0.4489901   2.7665193   0.41747993]]

 [[ 0.5183984   0.31333554 -0.81508636]
  [ 0.5432615   0.24723972 -0.7677377 ]
  [ 0.56130743  0.2474026  -0.7675525 ]
  ...
  [ 0.39947158  2.676335    0.976349  ]
  [ 0.6281753   2.7545614   0.49198413]
  [ 0.449516    2.7662194   0.4151463 ]]

 ...

 [[ 0.5184535   0.3373183  -0.8274746 ]
  [ 0.54278743  0.26486892 -0.77457756]
  [ 0.5594522   0.26353857 -0.7744814 ]
  ...
  [ 0.3999008   2.7135198   0.89675254]
  [ 0.636896    2.8050447   0.37377626

In [5]:
mediapipe1.close()
#mediapipe2.close()
cv2.destroyAllWindows()